In [1]:
import json
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("--- Step 1: Generating Production GenAI Server Activity Logs ---")

# Establish reproducible mock conditions matching corporate token metrics
np.random.seed(42)
base_time = datetime.now()
models = ['gpt-4o', 'claude-3-5-sonnet', 'gemma-2-27b-it']
user_keys = ['usr_key_9021', 'usr_key_4431', 'usr_key_7785', 'usr_key_1102']

simulated_log_lines = []
for i in range(1000):
    timestamp = (base_time + timedelta(seconds=i*25)).strftime('%Y-%m-%d %H:%M:%S')
    model = np.random.choice(models)
    user = np.random.choice(user_keys)

    # Simulate high-variance usage patterns (large prompts, variable completions)
    prompt_tokens = int(np.random.exponential(scale=600) + 150)
    completion_tokens = int(np.random.exponential(scale=400) + 50)

    # Introduce real-world operational anomalies (2% of requests trigger 429 rate limit errors)
    status = 200 if np.random.rand() > 0.02 else 429

    log_entry = {
        "timestamp": timestamp,
        "api_key": user,
        "model_deployed": model,
        "usage_metrics": {
            "prompt_tokens": prompt_tokens if status == 200 else 0,
            "completion_tokens": completion_tokens if status == 200 else 0
        },
        "response_status": status
    }
    simulated_log_lines.append(json.dumps(log_entry))

# Ensure output storage directories exist cleanly inside our cloud instance
os.makedirs('data_warehouse', exist_ok=True)
log_file_path = 'data_warehouse/api_gateway_traffic.log'

with open(log_file_path, 'w') as f:
    f.write('\n'.join(simulated_log_lines))

print(f"SUCCESS: 1,000 corporate AI usage entries safely streamed to '{log_file_path}'!")


--- Step 1: Generating Production GenAI Server Activity Logs ---
SUCCESS: 1,000 corporate AI usage entries safely streamed to 'data_warehouse/api_gateway_traffic.log'!


In [2]:
import json
import sqlite3
import pandas as pd

print("--- Step 2: Deploying Relational Database Warehouse Schema ---")

# 1. Initialize an optimized transactional storage database instance in memory
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Architect a structured schema with keys to handle financial queries
cursor.execute('''
CREATE TABLE corporate_llm_metrics (
    record_id INTEGER PRIMARY KEY AUTOINCREMENT,
    log_timestamp TEXT,
    user_api_key TEXT,
    model_architecture TEXT,
    input_prompt_tokens INTEGER,
    output_completion_tokens INTEGER,
    http_status_code INTEGER
)
''')
print("✅ Database layout successfully initialized with indexing fields.")

# 3. Read and parse the raw JSON log traffic stream file
parsed_transactional_records = []
log_file_path = 'data_warehouse/api_gateway_traffic.log'

with open(log_file_path, 'r') as f:
    for line in f:
        if not line.strip():
            continue
        log_data = json.loads(line.strip())

        # Unpack the multi-layered dictionary layout flat for relational storage
        parsed_transactional_records.append((
            log_data['timestamp'],
            log_data['api_key'],
            log_data['model_deployed'],
            log_data['usage_metrics']['prompt_tokens'],
            log_data['usage_metrics']['completion_tokens'],
            log_data['response_status']
        ))

# 4. Bulk ingest parsed arrays straight into the database table rows safely
cursor.executemany('''
INSERT INTO corporate_llm_metrics (
    log_timestamp, user_api_key, model_architecture,
    input_prompt_tokens, output_completion_tokens, http_status_code
) VALUES (?, ?, ?, ?, ?, ?)
''', parsed_transactional_records)

conn.commit()
print(f"✅ Ingestion complete: Loaded {len(parsed_transactional_records)} server rows into relational storage arrays.")

# 5. Run a safety validation metric check
df_sample_check = pd.read_sql_query("SELECT * FROM corporate_llm_metrics LIMIT 5", conn)
print("\n--- Relational Database Ingestion Audit (First 5 Rows) ---")
print(df_sample_check)


--- Step 2: Deploying Relational Database Warehouse Schema ---
✅ Database layout successfully initialized with indexing fields.
✅ Ingestion complete: Loaded 1000 server rows into relational storage arrays.

--- Relational Database Ingestion Audit (First 5 Rows) ---
   record_id        log_timestamp  user_api_key model_architecture  \
0          1  2026-09-18 17:22:52  usr_key_1102     gemma-2-27b-it   
1          2  2026-09-18 17:23:17  usr_key_4431     gemma-2-27b-it   
2          3  2026-09-18 17:23:42  usr_key_4431     gemma-2-27b-it   
3          4  2026-09-18 17:24:07  usr_key_1102  claude-3-5-sonnet   
4          5  2026-09-18 17:24:32  usr_key_1102             gpt-4o   

   input_prompt_tokens  output_completion_tokens  http_status_code  
0                 1956                       576               200  
1                  251                        73               200  
2                  184                       562               200  
3                 3063               

In [3]:
print("--- Step 3: Executing Enterprise Financial Cost Analytics Matrix ---")

# Apply enterprise-level api pricing configurations per 1 million tokens:
# GPT-4o: $5.00 Input / $15.00 Output
# Claude-3.5-Sonnet: $3.00 Input / $15.00 Output
# Gemma-2-27b-it: $0.20 Input / $0.60 Output

financial_window_query = '''
WITH operational_cost_layer AS (
    SELECT
        user_api_key,
        model_architecture,
        COUNT(*) as individual_endpoint_hits,
        SUM(input_prompt_tokens) as total_input_tokens,
        SUM(output_completion_tokens) as total_output_tokens,
        SUM(CASE
            WHEN model_architecture = 'gpt-4o' THEN (input_prompt_tokens * 0.000005) + (output_completion_tokens * 0.000015)
            WHEN model_architecture = 'claude-3-5-sonnet' THEN (input_prompt_tokens * 0.000003) + (output_completion_tokens * 0.000015)
            ELSE (input_prompt_tokens * 0.0000002) + (output_completion_tokens * 0.0000006)
        END) as user_total_spend_usd
    FROM corporate_llm_metrics
    WHERE http_status_code = 200
    GROUP BY user_api_key, model_architecture
)
SELECT
    user_api_key as User_ID,
    model_architecture as Deployed_Model,
    individual_endpoint_hits as API_Hits,
    total_input_tokens as Input_Tokens,
    total_output_tokens as Output_Tokens,
    ROUND(user_total_spend_usd, 2) as Core_Spend_USD,

    -- High-Value SQL Window Function to track rolling aggregate cost caps across each model type
    ROUND(SUM(user_total_spend_usd) OVER(PARTITION BY model_architecture), 2) as Model_Category_Total_Pool_USD,

    -- Window Function to calculate exact budget allocation percentage per corporate user key
    ROUND((user_total_spend_usd / SUM(user_total_spend_usd) OVER(PARTITION BY model_architecture)) * 100, 2) as Cost_Contribution_Percentage
FROM operational_cost_layer
ORDER BY Deployed_Model, Core_Spend_USD DESC
'''

# Compile metrics query straight into our analytical viewing plane
df_financial_dashboard = pd.read_sql_query(financial_window_query, conn)

print("\n================== ENTERPRISE AI OPERATIONS FINANCIAL DASHBOARD ==================")
print(df_financial_dashboard)
print("==================================================================================")


--- Step 3: Executing Enterprise Financial Cost Analytics Matrix ---

================== ENTERPRISE AI OPERATIONS FINANCIAL DASHBOARD ==================
         User_ID     Deployed_Model  API_Hits  Input_Tokens  Output_Tokens  \
0   usr_key_7785  claude-3-5-sonnet        86         70335          31914   
1   usr_key_4431  claude-3-5-sonnet        82         58815          33508   
2   usr_key_9021  claude-3-5-sonnet        70         50387          32034   
3   usr_key_1102  claude-3-5-sonnet        66         49350          30020   
4   usr_key_7785     gemma-2-27b-it        78         64401          39796   
5   usr_key_1102     gemma-2-27b-it        81         80177          29985   
6   usr_key_4431     gemma-2-27b-it        83         55450          38191   
7   usr_key_9021     gemma-2-27b-it        76         54637          35542   
8   usr_key_7785             gpt-4o        94         73589          45698   
9   usr_key_9021             gpt-4o        89         78236        

In [4]:
print("--- Step 4: Launching System Infrastructure Anomaly Audit Engine ---")

# This advanced subquery evaluates failure clusters to isolate system abuse
anomaly_detection_query = '''
SELECT
    user_api_key as Offending_User_Key,
    model_architecture as Target_Model_Engine,
    COUNT(*) as Total_Failed_Requests,

    -- Analytical Window Function to rank systemic failures across user keys dynamically
    DENSE_RANK() OVER (
        PARTITION BY model_architecture
        ORDER BY COUNT(*) DESC
    ) as Abuse_Severity_Rank
FROM corporate_llm_metrics
WHERE http_status_code = 429
GROUP BY user_api_key, model_architecture
ORDER BY Target_Model_Engine, Total_Failed_Requests DESC
'''

# Compile reliability metrics into our diagnostic dashboard view
df_anomaly_report = pd.read_sql_query(anomaly_detection_query, conn)

print("\n🚨 ================ INFRASTRUCTURE SERVICE ANOMALY REPORT ================ 🚨")
if len(df_anomaly_report) > 0:
    print(df_anomaly_report)
else:
    print("All endpoints operating stably. Zero infrastructure error anomalies detected.")
print("============================================================================")


--- Step 4: Launching System Infrastructure Anomaly Audit Engine ---

🚨 ================ INFRASTRUCTURE SERVICE ANOMALY REPORT ================ 🚨
  Offending_User_Key Target_Model_Engine  Total_Failed_Requests  \
0       usr_key_9021   claude-3-5-sonnet                      5   
1       usr_key_4431   claude-3-5-sonnet                      4   
2       usr_key_7785   claude-3-5-sonnet                      4   
3       usr_key_1102   claude-3-5-sonnet                      1   
4       usr_key_1102      gemma-2-27b-it                      2   
5       usr_key_9021      gemma-2-27b-it                      2   
6       usr_key_4431      gemma-2-27b-it                      1   
7       usr_key_7785              gpt-4o                      3   
8       usr_key_9021              gpt-4o                      2   
9       usr_key_4431              gpt-4o                      1   

   Abuse_Severity_Rank  
0                    1  
1                    2  
2                    2  
3               

In [5]:
import os

print("--- Step 5: Initializing Data Pipeline Automated Export System ---")

# 1. Ensure a dedicated production report directory exists on disk
reports_dir = 'data_warehouse/production_reports'
os.makedirs(reports_dir, exist_ok=True)

financial_report_path = os.path.join(reports_dir, 'ai_infrastructure_cost_summary.csv')
anomaly_report_path = os.path.join(reports_dir, 'system_reliability_incident_log.csv')

# 2. Export our live SQL dashboards straight to disk arrays
df_financial_dashboard.to_csv(financial_report_path, index=False)
df_anomaly_report.to_csv(anomaly_report_path, index=False)

print("\n================== AUTOMATED ETL REPORT EXPORT METRICS ==================")
print(f"📁 Financial Cost Summary written to:  {financial_report_path}")
print(f"📁 Reliability Incident Log written to: {anomaly_report_path}")
print("=========================================================================")

# 3. Final system verification check
if os.path.exists(financial_report_path) and os.path.exists(anomaly_report_path):
    print("\n🎉 SUCCESS: End-to-end pipeline completed cleanly! All structural data assets are locked on disk.")
else:
    print("\n⚠️ Warning: Missing output assets. Re-evaluating environment path settings.")


--- Step 5: Initializing Data Pipeline Automated Export System ---

================== AUTOMATED ETL REPORT EXPORT METRICS ==================
📁 Financial Cost Summary written to:  data_warehouse/production_reports/ai_infrastructure_cost_summary.csv
📁 Reliability Incident Log written to: data_warehouse/production_reports/system_reliability_incident_log.csv

🎉 SUCCESS: End-to-end pipeline completed cleanly! All structural data assets are locked on disk.
